In [4]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime, time
import re

# --- CONFIGURATION ---
BASE_PATH_SPOT = r"C:\Users\RISHI\Downloads\nif1\nifty_data\nifty_spot\2024"
BASE_PATH_OPT = r"C:\Users\RISHI\Downloads\nif1\nifty_data\nifty_options\2024"
VIX_FILE_PATH = "INDIA VIX_minute.csv"
LOT_SIZE = 50
SL_TICKS = 25 
TARGET_PTS = 1000 
TRADING_START_TIME = time(9, 16)
SQUARE_OFF_TIME = time(15, 15)
RSI_PERIOD = 14

def get_atm_strike(spot_price):
    return int(round(spot_price / 50) * 50)

def parse_expiry_from_sym(symbol):
    """
    Robustly parses expiry. Handles cases where the day/year might be swapped.
    Ensures year is always 2020 or later.
    """
    try:
        # Regex to find: 2 digits, 3 letters (Month), 2 digits
        match = re.search(r'(\d{2})([A-Z]{3})(\d{2})', symbol)
        if not match: return None
        
        p1, month, p2 = match.groups()
        
        # Try Format: DDMMMYY (e.g., 04JAN24 -> Jan 4, 2024)
        try:
            dt = datetime.strptime(f"{p1}{month}{p2}", '%d%b%y').date()
            if dt.year >= 2020: return dt
        except: pass

        # Try Format: YYMMMDD (e.g., 24JAN04 -> Jan 4, 2024)
        try:
            dt = datetime.strptime(f"{p1}{month}{p2}", '%y%b%d').date()
            if dt.year >= 2020: return dt
        except: pass
    except:
        return None
    return None

def get_date_from_filename(filename):
    """Extracts date from filename. Returns None if no date found for clean sorting."""
    base = os.path.basename(filename)
    # Pattern: DD_MM_YYYY or DD-MM-YYYY
    m = re.search(r'(\d{1,2})[_-](\d{1,2})[_-](\d{4})', base)
    if m:
        try:
            d, month, y = map(int, m.groups())
            return datetime(y, month, d)
        except: return None
    # Pattern: YYYY_MM_DD
    m = re.search(r'(\d{4})[_-](\d{1,2})[_-](\d{1,2})', base)
    if m:
        try:
            y, month, d = map(int, m.groups())
            return datetime(y, month, d)
        except: return None
    return None

def get_option_file_path(dt):
    if dt is None: return None
    month_folder = str(dt.month)
    # Check common date formats in filenames
    for fmt in [dt.strftime('%d_%m_%Y'), dt.strftime('%d-%m-%Y'), dt.strftime('%Y-%m-%d')]:
        pattern = os.path.join(BASE_PATH_OPT, month_folder, f"*{fmt}*.csv")
        files = glob.glob(pattern)
        if files: return files[0]
    return None

def calculate_indicators(df):
    """
    Matches TradingView RSI (Wilder's RMA) and EMAs.
    """
    df = df.copy()
    df['ema5'] = df['low'].ewm(span=21, adjust=False).mean()
    df['ema8'] = df['low'].ewm(span=34, adjust=False).mean()
    df['ema8_high'] = df['high'].ewm(span=34, adjust=False).mean()
    
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    
    # alpha=1/N is the formula for Wilder's Smoothing (RMA)
    avg_gain = gain.ewm(alpha=1/RSI_PERIOD, adjust=True, min_periods=1).mean()
    avg_loss = loss.ewm(alpha=1/RSI_PERIOD, adjust=True, min_periods=1).mean()
    
    rs = np.divide(avg_gain, avg_loss, out=np.full_like(avg_gain, np.nan), where=avg_loss != 0)
    df['rsi'] = 100 - (100 / (1 + rs))
    
    df.loc[avg_loss == 0, 'rsi'] = 100
    df.loc[(avg_loss == 0) & (avg_gain == 0), 'rsi'] = 50
    return df

def run_backtest():
    all_trades = []
    
    # 1. Load VIX (Resampled to 5min)
    vix_5m = None
    try:
        dv = pd.read_csv(VIX_FILE_PATH)
        dv['datetime'] = pd.to_datetime(dv['date'])
        vix_5m = dv.set_index('datetime')['close'].resample('5min').last().ffill()
    except:
        print("Warning: INDIA VIX file not found. VIX filter will be ignored.")

    # 2. Collect and Filter Spot Files
    raw_spot_files = []
    for m in range(1, 13):
        p = os.path.join(BASE_PATH_SPOT, str(m))
        if os.path.exists(p): raw_spot_files.extend(glob.glob(os.path.join(p, "*.csv")))
    
    # Sort files chronologically by extracting actual datetime
    spot_files = []
    for f in raw_spot_files:
        dt = get_date_from_filename(f)
        if dt: spot_files.append((dt, f))
    spot_files.sort()

    # 3. Process Each Day
    for i, (current_dt, s_file) in enumerate(spot_files):
        try:
            current_date = current_dt.date()
            df_spot = pd.read_csv(s_file)
            
            # Find ATM at Market Open
            open_candle = df_spot[df_spot['time'] >= '09:15:00'].head(1)
            if open_candle.empty: continue
            atm_strike = get_atm_strike(open_candle['close'].values[0])
            
            # Load Today's Options
            opt_path = get_option_file_path(current_dt)
            if not opt_path: continue
            df_opt_today = pd.read_csv(opt_path)
            
            # Load Previous Day for RSI Warm-up
            df_opt_prev = None
            if i > 0:
                prev_dt, _ = spot_files[i-1]
                prev_path = get_option_file_path(prev_dt)
                if prev_path: df_opt_prev = pd.read_csv(prev_path)

            contracts = {}
            for side in ['CE', 'PE']:
                strike_str = str(atm_strike)
                matches = [s for s in df_opt_today['symbol'].unique() if strike_str in s and side in s]
                
                # Pick Nearest Valid Expiry
                valid_expiries = []
                for s in matches:
                    exp = parse_expiry_from_sym(s)
                    if exp and exp >= current_date:
                        valid_expiries.append((s, exp))
                
                if not valid_expiries: continue
                sel_sym = sorted(valid_expiries, key=lambda x: x[1])[0][0]

                def get_1m_data(src, sym):
                    d = src[src['symbol'] == sym].copy()
                    if d.empty: return d
                    d['dt'] = pd.to_datetime(d['date'] + ' ' + d['time'])
                    return d.set_index('dt').sort_index()

                # Merge today with yesterday to ensure RSI doesn't start at 0
                today_1m = get_1m_data(df_opt_today, sel_sym)
                if today_1m.empty: continue
                
                combined_1m = today_1m
                if df_opt_prev is not None:
                    prev_1m = get_1m_data(df_opt_prev, sel_sym)
                    if not prev_1m.empty:
                        combined_1m = pd.concat([prev_1m, today_1m]).sort_index()

                # Resample and Calc Indicators
                df_5m = combined_1m.resample('5min').agg({'open':'first','high':'max','low':'min','close':'last'}).dropna()
                processed = calculate_indicators(df_5m)
                
                contracts[side] = {
                    'sym': sel_sym,
                    'df': processed[processed.index.date == current_date],
                    'orb_low': today_1m.resample('5min').agg({'low':'min'}).iloc[0]['low']
                }

            # 4. Trading Logic
            if not contracts: continue
            ts_list = sorted(list(set().union(*(c['df'].index for c in contracts.values()))))
            active_trade = None
            ce_done = pe_done = False

            for t in ts_list:
                curr_t = t.time()
                if active_trade:
                    side = active_trade['side']
                    if t not in contracts[side]['df'].index: continue
                    curr = contracts[side]['df'].loc[t]
                    
                    exit_trig, reason, price = False, "", 0
                    if curr_t >= SQUARE_OFF_TIME: exit_trig, reason, price = True, "Time Exit", curr['close']
                    elif curr['low'] <= active_trade['Target']: exit_trig, reason, price = True, "Target", active_trade['Target']
                    elif curr['high'] >= active_trade['SL']: exit_trig, reason, price = True, "SL", active_trade['SL']
                    elif curr['close'] > curr['ema8_high']: exit_trig, reason, price = True, "EMA Exit", curr['close']
                    
                    if exit_trig:
                        all_trades.append({
                            'Date': str(current_date), 'Symbol': active_trade['sym'],
                            'Entry Time': active_trade['Entry Time'], 'Exit Time': t.strftime('%H:%M:%S'),
                            'Entry': active_trade['Entry Price'], 'Exit': price,
                            'PnL': (active_trade['Entry Price'] - price) * LOT_SIZE, 'Result': reason
                        })
                        active_trade = None
                    else:
                        # Trailing SL Logic
                        profit = active_trade['Entry Price'] - curr['low']
                        if profit >= 1:
                            active_trade['SL'] = max(active_trade['SL'], (active_trade['Entry Price'] + SL_TICKS)) - (int(profit) * 0.5)
                
                else:
                    if TRADING_START_TIME <= curr_t < SQUARE_OFF_TIME:
                        vx = vix_5m.get(t)
                        for side in ['CE', 'PE']:
                            if (side == 'CE' and ce_done) or (side == 'PE' and pe_done): continue
                            if side not in contracts or t not in contracts[side]['df'].index: continue
                            
                            c = contracts[side]['df'].loc[t]
                            # Entry conditions (Shorting example based on your original logic)
                            if (vx < 15 and c['close'] < c['ema5'] and c['close'] < c['ema8'] and 
                                c['close'] < contracts[side]['orb_low'] and c['rsi'] >= 30 and c['rsi'] < 60):
                                
                                active_trade = {
                                    'side': side, 'sym': contracts[side]['sym'],
                                    'Entry Price': c['close'], 'Entry Time': t.strftime('%H:%M:%S'),
                                    'SL': c['close'] + SL_TICKS, 'Target': c['close'] - TARGET_PTS
                                }
                                if side == 'CE': ce_done = True
                                else: pe_done = True
                                break


        except Exception as e:
            print(f"Error on {current_dt}: {e}")

    # Save Results
    if all_trades:
        pd.DataFrame(all_trades).to_excel("Nifty_Backtest_Full_Year.xlsx", index=False)
        print("\nSuccess! Trades saved to Nifty_Backtest_Full_Year.xlsx")
    else:
        print("\nNo trades were triggered. Verify data paths and entry conditions.")

if __name__ == "__main__":
    run_backtest()


Success! Trades saved to Nifty_Backtest_Full_Year.xlsx


In [5]:
import pandas as pd

# Load the backtest results file
file_path = r'Nifty_Backtest_Full_Year.xlsx'
df = pd.read_excel(file_path)

# Convert PnL to numeric if it's not
df['PnL'] = pd.to_numeric(df['PnL'], errors='coerce')

# Calculate Summary
total_profit = df['PnL'].sum()
no_of_trades = len(df)
wins = len(df[df['PnL'] >= 0])
losses = len(df[df['PnL'] < 0])
win_rate = (wins / no_of_trades) * 100

# Calculate Drawdown
df['CumPnL'] = df['PnL'].cumsum()
df['MaxCum'] = df['CumPnL'].cummax()
df['Drawdown'] = df['MaxCum'] - df['CumPnL']
max_drawdown = df['Drawdown'].max()

# Sample of trades
sample_trades = df[['Date', 'Symbol', 'PnL']].head(10)

print(f"Total Profit: {total_profit}")
print(f"Total Trades: {no_of_trades}")
print(f"Max Drawdown: {max_drawdown}")
print("\nSample Trades:")
print(sample_trades)
print(win_rate)

Total Profit: 41792.5
Total Trades: 213
Max Drawdown: 8327.5

Sample Trades:
         Date               Symbol     PnL
0  2024-01-01  NIFTY04JAN2421700CE -1200.0
1  2024-01-01  NIFTY04JAN2421700PE  1172.5
2  2024-01-02  NIFTY04JAN2421700CE   350.0
3  2024-01-03  NIFTY04JAN2421600CE  2717.5
4  2024-01-04  NIFTY04JAN2421600PE  1990.0
5  2024-01-05  NIFTY11JAN2421750PE -1225.0
6  2024-01-05  NIFTY11JAN2421750CE  -567.5
7  2024-01-08  NIFTY11JAN2421750PE -1225.0
8  2024-01-08  NIFTY11JAN2421750CE -1062.5
9  2024-01-09  NIFTY11JAN2421650PE  -107.5
44.13145539906103


In [3]:
pd.set_option('display.max_rows', None)
df


,Date,Symbol,Entry Time,Exit Time,Entry,Exit,PnL,Result,CumPnL,MaxCum,Drawdown
0,2024-01-01,NIFTY04JAN2421700CE,09:35:00,09:50:00,124.40,148.40,-1200.0,SL,-1200.0,-1200.0,0.0
1,2024-01-01,NIFTY04JAN2421700PE,10:30:00,15:10:00,91.80,68.35,1172.5,EMA Exit,-27.5,-27.5,0.0
2,2024-01-02,NIFTY04JAN2421700CE,09:55:00,14:05:00,85.00,78.00,350.0,EMA Exit,322.5,322.5,0.0
3,2024-01-03,NIFTY04JAN2421600CE,09:35:00,15:15:00,75.40,21.05,2717.5,Time Exit,3040.0,3040.0,0.0
4,2024-01-04,NIFTY04JAN2421600PE,09:20:00,15:15:00,39.85,0.05,1990.0,Time Exit,5030.0,5030.0,0.0
5,2024-01-05,NIFTY11JAN2421750PE,09:50:00,10:45:00,113.55,138.05,-1225.0,SL,3805.0,5030.0,1225.0
6,2024-01-05,NIFTY11JAN2421750CE,11:00:00,12:20:00,95.75,107.10,-567.5,EMA Exit,3237.5,5030.0,1792.5
7,2024-01-08,NIFTY11JAN2421750PE,09:20:00,09:30:00,101.60,126.10,-1225.0,SL,2012.5,5030.0,3017.5
8,2024-01-08,NIFTY11JAN2421750CE,09:35:00,10:30:00,71.75,93.00,-1062.5,EMA Exit,950.0,5030.0,4080.0
9,2024-01-09,NIFTY11JAN2421650PE,10:15:00,14:05:00,69.45,71.60,-107.5,EMA Exit,842.5,5030.0,4187.5
